In [2]:
# 01_neo4j_embed_sentence_transformers.ipynb
# -------------------------------------------
# Neo4j → Text → SentenceTransformer embeddings
# -------------------------------------------

from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv
import os

# -------- Config --------
load_dotenv()

URI = "neo4j://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = os.getenv("NEO4J_PASSWORD")

MODEL_NAME = "all-MiniLM-L6-v2"

BASE_DIR = Path("graph")
EMB_DIR = BASE_DIR / "embeddings"
DATA_DIR = BASE_DIR / "data"

EMB_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# -------- Connect Neo4j --------
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# -------- Cypher Queries --------
QUERIES = [
    """
    MATCH (d:Disease)-[:MANIFESTS_AS]->(s:Symptom)
    RETURN d.name_en AS source, 'manifests as' AS relation, s.name_en AS target
    """,
    """
    MATCH (d:Disease)-[:TREATED_BY]->(t:Treatment)
    RETURN d.name_en AS source, 'treated by' AS relation, t.name_en AS target
    """,
    """
    MATCH (h:Herb)-[:BENEFICIAL_FOR]->(d:Disease)
    RETURN h.name_en AS source, 'beneficial for' AS relation, d.name_en AS target
    """,
    """
    MATCH (d:Disease)-[:CAUSED_BY]->(n:Nidana)
    RETURN d.name_en AS source, 'caused by' AS relation, n.name_en AS target
    """
]

# -------- Extract Knowledge --------
facts = []

with driver.session(database="neo4j") as session:
    for q in QUERIES:
        for r in session.run(q):
            text = f"{r['source']} {r['relation']} {r['target']}."
            facts.append(text)

print("Total facts:", len(facts))

# -------- Save raw facts --------
pd.DataFrame({"text": facts}).to_json(
    DATA_DIR / "neo4j_facts.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# -------- Embed --------
model = SentenceTransformer(MODEL_NAME)

embeddings = model.encode(
    facts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

np.save(EMB_DIR / "neo4j_st.npy", embeddings)

# -------- Metadata --------
meta_df = pd.DataFrame({
    "text": facts,
    "source": "neo4j",
    "embedding_model": MODEL_NAME
})

meta_df.to_parquet(EMB_DIR / "metadata.parquet")

print("SentenceTransformer embeddings saved")

Total facts: 48


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.06it/s]


SentenceTransformer embeddings saved
